Exercise 1- Conceptual Questions

1. The SARSA Quintuple
In SARSA, the update uses (S, A, R, S', A'). The next action A' is essential because SARSA is an on-policy algorithm. This means it learns the value of the policy that it actually follows. Since the agent chooses A' using its current policy (including exploration), the update reflects the real behavior of the agent, not an optimal one


2. Choosing an Algorithm
I would choose SARSA. SARSA is more conservative because it considers the action actually taken (including risky exploratory moves). This leads to safer policies, which is important when mistakes can cause damage. Q-Learning, on the other hand, is more aggressive and assumes optimal future actions


3. DQN Input/Output
The input of the neural network would be an image of the racetrack.
Input shape: (height, width, channels), for example (84, 84, 3).
The output would be Q-values for each action:
['accelerate', 'brake', 'turn_left', 'turn_right'].

Output shape: (4,)

Each output value represents the expected future reward for that action

In [2]:
import gymnasium as gym
import numpy as np

def sarsa(env, alpha, gamma, epsilon, episodes):
    state_size = env.observation_space.n
    action_size = env.action_space.n
    
    Q = np.zeros((state_size, action_size))
    
    for episode in range(episodes):
        state, _ = env.reset()
        
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])
        
        done = False
        
        while not done:
            new_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            if np.random.rand() < epsilon:
                new_action = env.action_space.sample()
            else:
                new_action = np.argmax(Q[new_state])
            
            Q[state, action] = Q[state, action] + alpha * (
                reward + gamma * Q[new_state, new_action] - Q[state, action]
            )
            
            state = new_state
            action = new_action
    
    return Q

In [3]:
env = gym.make('FrozenLake-v1', is_slippery=False)

alpha = 0.5
gamma = 0.9
epsilon = 0.2
episodes = 15000

Q_sarsa = sarsa(env, alpha, gamma, epsilon, episodes)

print("SARSA Q-table:")
print(Q_sarsa)

env.close()

SARSA Q-table:
[[0.17552752 0.16350069 0.22884338 0.21032956]
 [0.19102574 0.         0.32890779 0.28818174]
 [0.1219529  0.51310706 0.23741977 0.39074705]
 [0.23268792 0.         0.17075706 0.20031286]
 [0.11299933 0.09743839 0.         0.23235642]
 [0.         0.         0.         0.        ]
 [0.         0.54565267 0.         0.3061684 ]
 [0.         0.         0.         0.        ]
 [0.07029605 0.         0.12872133 0.1877016 ]
 [0.46282229 0.37517621 0.66345505 0.        ]
 [0.5329234  0.83678289 0.         0.23875792]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.24590871 0.89988638 0.57618304]
 [0.76774357 0.89961617 1.         0.6057204 ]
 [0.         0.         0.         0.        ]]


SARSA vs Q-Learning

SARSA learns a more conservative policy because it updates using the actual action taken.

Q-Learning learns more aggressive values because it always assumes the best possible action in the future.

As a result:
- Q-Learning usually learns faster and gives higher values.
- SARSA produces safer and more stable policies.

Exercise 3- Designing a DQN

1. Network Architecture

Input layer size: 4 (cart position, velocity, pole angle, pole velocity)

Output layer size: 2 (left, right)

Architecture:
- Input layer: 4 neurons
- Hidden layer 1: 24 neurons (ReLU)
- Hidden layer 2: 24 neurons (ReLU)
- Output layer: 2 neurons (linear activation)


2. Experience Replay

We use a replay buffer (a list or deque).

Each experience stores:
(state, action, reward, next_state, done)


3. Learning Step

1. Sample a random mini-batch from the replay buffer.
2. For each sample, compute the target:
   target = reward + gamma * max(Q(next_state)) (if not done)
3. Compare predicted Q-values with target.
4. Update the network using gradient descent.

This stabilizes training and reduces correlation between samples.